In [2]:
# packages
using Markdown
using InteractiveUtils
using NonlinearSolve
using Plots
using StaticArrays
# files
using PKAssetPrices

[ Info: Precompiling PKAssetPrices [5fa4aa55-4e4c-4901-99b7-08200d3ce2d7] (cache misses: wrong dep version loaded (2), include_dependency fhash change (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: PKAssetPrices.BalanceSheetAbstractData(:Private, [:deposits, :loans], [:deposits], [:loans], Dict{Symbol, Union{Expr, Symbol}}(:loans => :dL, :deposits => :dM))
[ Info: PKAssetPrices.BalanceSheetAbstractData(:Banks, [:loans, :reserves, :deposits, :central_bank_credit], [:loans, :reserves], [:deposits, :central_bank_credit], Dict{Symbol, Union{Expr, Symbol}}(:loans => :dL, :central_bank_credit => :dR, :reserves => :dR, :deposits => :dM))
[ Info: PKAssetPrices.BalanceSheetAbstractData(:CentralBank, [:central_bank_credit, :reserves], [:central_bank_credit], [:reserves], Dict{Symbol, Union{Expr, Symbol}}(:central_bank_credit => :dR, :reserves => :dR))
[ Info: Expr[:(Y::Float64), :(ND::Float64), :(D::Float64), :(r::Float64), :(i::

# Defining models

A model can be easily defined via the @model macro, one must define @variables, @parameters and @equations

In [3]:
@model SimplePK2 begin
	@variables begin
		Y = "Output"
		ND = "Non debt-financed demand"
		D = "debt-financed demand"
		r = "interest rate"
		i = "policy rate"
		P = "price level"
		dL = "change in loans"
		dM = "change in money"
		dR = "change in reserves"
		W = "wage level"
		N = "employment"
		U = "unemployment rate"
	end

	@parameters begin
		b = 0.5, "consumption rate"
		c = 0.8, "credit rationing"
		d₀ = 5.0, "autonomuous  credit financed demand"
		d₁ = 8, "induced credit financed demand"
		i₀ = 0.01, "autonomuous policy rate"
		i₁ = 0.05, "inflation infuced policy rate"
		m = 0.15, "policy rate markup"
		k = 0.3, "resere share"
		n = 0.15, "firm markup"
		W₀ = 2.0, "autonomuous wages"
		h = 0.8, "bargaining power?"
		a = 0.8, "production induced employment "
		Nᶠ = 12.0, "total labour supply"
	end

	@equations begin
		Y == ND + c * D
		ND == b * Y
		D == d₀ - d₁ * r
		r == (1 + m) * i
		i == i₀ + i₁ * P
		dL == c * D
		dM == dL
		dR == k * dM
		P == (1 + n) * a * W
		W == W₀ - h * U
		N == a * Y
		U == 1 - N / Nᶠ
	end

	@curves begin
		IS(r) = (1/(1-b)) * (c * (d₀ - d₁ * r))
		IR(Y) = (1 + m) * (i₀ + i₁ * (1 + n) * a * (W₀ - h * (1 - (a * Y) / Nᶠ)))
		AD(P) = (1/(1-b)) * (c * (d₀ - d₁ * ((1 + m) * (i₀ + i₁ * P))))
		AS(Y) = (1 + n) * a * (W₀ - h * (1 - (a * Y) / Nᶠ))
	end

end
  

[ Info: Expr[:(Y::Float64), :(ND::Float64), :(D::Float64), :(r::Float64), :(i::Float64), :(P::Float64), :(dL::Float64), :(dM::Float64), :(dR::Float64), :(W::Float64), :(N::Float64), :(U::Float64)]


Any[]

# Defining scenarios

With the @scenario macro you can easily set different parameters, the name must be the same as the model defined via the @model macro

In [4]:
scen1 = @scenario SimplePK begin
    b = 0.6
end

SimplePKModel(SimplePKParams(0.6, 0.8, 5.0, 8.0, 0.01, 0.05, 0.15, 0.3, 0.15, 2.0, 0.8, 0.8, 12.0), [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])

Das Modell kann folgend gelöst werden

In [5]:
sol = PKAssetPrices.solve_model(scen1)

PKAssetPrices.SimplePKSolution(PKAssetPrices.SimplePKSolutionVar(8.420220040415586, 5.05213202424935, 4.210110020207794, 0.09873624747402576, 0.08585760649915297, 1.517152129983058, 3.3680880161662348, 3.3680880161662348, 1.0104264048498703, 1.649078402155498, 6.7361760323324695, 0.43865199730562754), BalanceSheet[PKAssetPrices.SimplePKPrivate(3.3680880161662348, 3.3680880161662348), PKAssetPrices.SimplePKBanks(3.3680880161662348, 1.0104264048498703, 3.3680880161662348, 1.0104264048498703), PKAssetPrices.SimplePKCentralBank(1.0104264048498703, 1.0104264048498703)])

Mit .sol.variable_name kann auf die gelöste Variable zugegriffen werden

In [6]:
sol.sol.Y

8.420220040415586

In [7]:
sol.sheets[2].reserves

1.0104264048498703

In [8]:
plot(x -> PKAssetPrices.IS(x,SimplePKParams()))

LoadError: #5 is not a Function, or is not defined at any of the values [-5.0, -1.0, 0.0, 0.01]

## Balance Sheets
Es ist auch leicht möglich balance sheets zu generieren (siehe simplemodel.jl), diese werden automatisch befüllt und können über sol.sheets abgerufen werden

In [9]:
PKAssetPrices.display_balance_sheet.(sol.sheets)


  Balance Sheet
ASSETS                         | LIABILITIES                   
------------------------------------------------------------
Deposits: 3.37                 | Loans: 3.37                   
------------------------------------------------------------
TOTAL: 3.37                    | TOTAL: 3.37                   

Net Worth: 0.0

  Balance Sheet
ASSETS                         | LIABILITIES                   
------------------------------------------------------------
Loans: 3.37                    | Central Bank Credit: 1.01     
Reserves: 1.01                 | Deposits: 3.37                
------------------------------------------------------------
TOTAL: 4.38                    | TOTAL: 4.38                   

Net Worth: 0.0

  Balance Sheet
ASSETS                         | LIABILITIES                   
------------------------------------------------------------
Central Bank Credit: 1.01      | Reserves: 1.01                
------------------------------------

3-element Vector{Nothing}:
 nothing
 nothing
 nothing

In [10]:
sum(PKAssetPrices.total_liabilities.(sol.sheets)) - sum(PKAssetPrices.total_assets.(sol.sheets))

2.357661611316365

### Alte Sachen, die nicht so recht geholfen haben ein dynamisches Environment bereitzustellen

In [16]:
using PyCall

@pyimport ipywidgets as widgets
@pyimport IPython.display as Pdisplay

# Create sliders
c_slider = widgets.FloatSlider(min=0, max=1, step=0.01, value=0.5, description="c")
b_slider = widgets.FloatSlider(min=0, max=1, step=0.01, value=0.5, description="b")
d0_slider = widgets.FloatSlider(min=0, max=10, step=0.1, value=5.0, description="d0")

# Display
Pdisplay.display(widgets.VBox([c_slider, b_slider, d0_slider]))

ArgumentError: ArgumentError: Package PyCall not found in current path.
- Run `import Pkg; Pkg.add("PyCall")` to install the PyCall package.

In [17]:
using IJulia
using PyCall

Pwidgets = pyimport("ipywidgets")

# Create sliders
c_slider = Pwidgets.FloatSlider(min=0, max=1, step=0.01, value=0.5, description="c")
b_slider = Pwidgets.FloatSlider(min=0, max=1, step=0.01, value=0.5, description="b")
d0_slider = Pwidgets.FloatSlider(min=0, max=10, step=0.1, value=5.0, description="d0")

# Display (this should work in IJulia)
vbox=display(Pwidgets.VBox([c_slider, b_slider, d0_slider]))

Main.IJulia.display(vbox)
display(vbox)

ArgumentError: ArgumentError: Package IJulia not found in current path.
- Run `import Pkg; Pkg.add("IJulia")` to install the IJulia package.

In [18]:
using WGLMakie, Makie

a = Observable(1)
b = Observable(1)

f(a, b) = sin(a*b)

fig = Figure()

ax = Axis(fig[1, 1])
lines!(ax, 0..10, x -> sin(a[] * x))

sl1 = Slider(fig[2, 1], range=1:5, startvalue=1)
sl2 = Slider(fig[3, 1], range=1:5, startvalue=1)

on(sl1.value) do v
    a[] = v
end

on(sl2.value) do v
    b[] = v
end

fig

ArgumentError: ArgumentError: Package WGLMakie not found in current path.
- Run `import Pkg; Pkg.add("WGLMakie")` to install the WGLMakie package.

In [19]:
f(a, b) = sin(a*b)

@bind a Slider(1:5)
@bind b Slider(1:5)

f(a, b)


LoadError: LoadError: UndefVarError: `@bind` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /home/franzs/Schreibtisch/Arbeit/PKAssetPrices/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:3

In [20]:
begin
	plot(range(0,10,10),y -> as_curve(y,params), label = "AS", title = "AS-AD curve")
	plot!(range(0.10,10), p -> ad_curve(p,params), label = "AD")
end

UndefVarError: UndefVarError: `as_curve` not defined in `Main`
Suggestion: check for spelling errors or missing imports.